
# Handoff Pattern
**Two Handoff patterns** are implemented:

| # | Pattern | Mode | Structure | Cell |
|---|---|---|---|---|
| 1 | **Research & Summary Workflow** | autonomous | `coordinator` → `research_agent` / `summary_agent` | 10-18 |
| 2 | **Customer Support (Interactive)** | human-in-the-loop | `triage` → `order_specialist` / `return_specialist` / `billing_specialist` (without `with_autonomous_mode()`) | 20-26 |

Essentially, it is a **combination of two axes**:

- **Agent configuration**: Research & Summary vs Customer Support
- **Execution mode**: autonomous (self-completing) vs human-in-the-loop (waiting for user input)

```
              autonomous            human-in-the-loop
Research      Pattern 1              (Not implemented in this notebook)
Support       (Not implemented in this notebook)  Pattern 2
```

In [ ]:
import logging  
from typing import Any, List  
import os  
from dotenv import load_dotenv  

from agent_framework.azure import AzureOpenAIChatClient
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.orchestrations import HandoffBuilder

# import warnings
# warnings.simplefilter('ignore')

load_dotenv(override=True)

## Setting Up Tracer with OpenTelemetry
Using OpenTelemetry tracer is convenient for debugging multi-agent systems.
Since `setup_observability` is not provided in this environment's Agent Framework,
we manually set up the OpenTelemetry `TracerProvider` (including Exporter/Processor) and enable instrumentation with `enable_instrumentation()`.
Here we use OTLP gRPC (e.g., Jaeger / OpenTelemetry Collector port `4317`) as the trace destination.

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# Configure via environment variables (Agent Framework reads standard OTEL_* vars)
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Disable Metrics (change as needed)

# Specify enable_sensitive_data=True to enable collection of sensitive data (OpenAI IN/OUT data)
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

In [ ]:
mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Authentication Method Selection ===
# True: API Key authentication, False: DefaultAzureCredential authentication
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 Authentication: API Key")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework automatically reads the AZURE_OPENAI_API_KEY environment variable
    # and prioritizes it over credential, so we explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 Authentication: DefaultAzureCredential")

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## Tools の定義(MCP Streamable HTTP)

In [ ]:
# Create MCP Streamable HTTP tools (set longer timeout as initial connection may be slow)
mcp_tool = MCPStreamableHTTPTool(
    name="mcp_tools",
    url=mcp_server_uri,
    description="A tool for retrieving game shop products, orders, inventory, user information, and Twitter analytics data.",
    timeout=120,
)

mcp_microsoft_learn_tool = MCPStreamableHTTPTool(
    name="mcp_microsoft_learn_tools",
    url="https://learn.microsoft.com/api/mcp",
    description="A tool for retrieving Microsoft Learn Docs. Provides the latest information from official Microsoft documentation.",
    timeout=120,
)

# Create Azure OpenAI client
chat_client = AzureOpenAIChatClient(
    **auth_kwargs,
    endpoint=azure_openai_endpoint,
    api_version=api_version,
    deployment_name=azure_deployment,
)

print(f"MCP Tool: {mcp_tool}")

In [ ]:
from __future__ import annotations
import sys
from contextlib import nullcontext
from typing import Optional, cast
from agent_framework import AgentResponse, AgentResponseUpdate, Message, WorkflowEvent

async def run_stream_pretty(
    workflow,
    task: str,
    *,
    tracer=None,
    span_name: str = "Handoff",
    print_final: bool = True,
) -> list[Message]:
    """
    
    Executes workflow.run(task, stream=True) and displays streaming output
    in real-time on Jupyter Notebook.

    Prerequisites (when using GroupChatBuilder):
      Build with GroupChatBuilder(..., intermediate_outputs=True).
      By default (False), only the orchestrator's output is yielded,
      and participant AgentResponseUpdate events are filtered.

    Display:
      - AgentResponseUpdate → Display tokens incrementally (sys.stdout.write + flush)
      - On executor switch → newline + name header
      - Final list[Message] → Display as CONVERSATION LOG summary

    Notes:
      ★ Uses start_as_current_span() to properly nest workflow internal spans
      as children of this span.
      _process_events is a regular async function, not an async generator,
      so no GeneratorExit issues occur when combined with context managers.
    """
    final_conversation: list[Message] = []
    last_executor_id: Optional[str] = None

    def _write(text: str) -> None:
        """Write with guaranteed flush, even in Jupyter"""
        sys.stdout.write(text)
        sys.stdout.flush()

    async def _process_events(workflow, task):
        nonlocal final_conversation, last_executor_id
        async for event in workflow.run(task, stream=True):
            if not isinstance(event, WorkflowEvent):
                continue

            # ----- executor switch notification -----
            if event.type == "executor_invoked":
                eid = event.executor_id
                if eid != last_executor_id:
                    if last_executor_id is not None:
                        _write("\n")
                    _write(f"🤖 {eid}: ")
                    last_executor_id = eid

            # ----- output event -----
            elif event.type == "output":
                data = event.data

                # (1) ストリーミングチャンク: AgentResponseUpdate
                if isinstance(data, AgentResponseUpdate):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    chunk = data.text or ""
                    if chunk:
                        _write(chunk)

                # (2) Non-streaming response: AgentResponse
                elif isinstance(data, AgentResponse):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    text = data.text or ""
                    if text:
                        _write(text)

                # (3) Final conversation log: list[Message]
                elif isinstance(data, list):
                    final_conversation = cast(list[Message], data)

    # ★ Use start_as_current_span() to nest workflow internal spans as children.
    #   _process_events is a regular async function, so no context conflicts occur.
    cm = tracer.start_as_current_span(span_name) if tracer else nullcontext()
    with cm:
        await _process_events(workflow, task)

    # Newline at end of stream
    _write("\n")

    if print_final and final_conversation:
        print("\n" + "=" * 80)
        print("CONVERSATION LOG")
        print("=" * 80)
        for msg in final_conversation:
            author = getattr(msg, "author_name", None) or msg.role
            text = msg.text or str(msg)
            print(f"\n[{author}]")
            print(text)
            print("-" * 80)

    return final_conversation

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    ワークフローのグラフ情報を出力し、SVGで保存し、プレビューする関数
    
    Args:
        workflow: Workflow object to visualize
        filename: Filename to save (without extension)
        show_preview: Whether to show the preview
    
    Returns:
        Path to the saved SVG file
    """
    # Create WorkflowViz object
    viz = WorkflowViz(workflow)
    
    # 3. Save as SVG file
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"✅ SVG file saved: {svg_path}")
        print("=" * 60)
        print()
        
        # 4. Display SVG preview
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("❌ Error: 'graphviz' package is not installed")
        print("Install with: pip install agent-framework[viz] --pre")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"❌ An error occurred: {e}")
        return None

## Agent Definition
### Handoff Parameters
"With Handoff, you specify which other agents each agent can delegate to. Since there is no manager like a Selector to determine the transition target, the decision of who to hand off to depends on each agent's prompt."

In [ ]:
# 3. -----------------  Agent Definitions -----------------  
coordinator = Agent(
    name="coordinator",
    client=chat_client,
    description="An agent that plans tasks. Routes user requests to the appropriate specialist agents.",
    instructions=(
"""
You are a coordinator. You decompose user queries into research tasks and summary tasks.
Assign the two tasks to the appropriate specialists in order.
Once all research and summarization is complete, respond to the user as FINAL_RESPONSE.
"""
    )
)

research_agent = Agent(
    name="research_agent",
    client=chat_client,
    description="research_agent",
    tools=[mcp_tool, mcp_microsoft_learn_tool],
    instructions=(
"""
You are a research specialist who thoroughly investigates topics on the Microsoft Learn site
- When given a research topic, break it down into multiple aspects and explore each one
- Continue research across multiple responses. Do not try to finish everything in a single response
- After each response, consider whether there are other aspects to explore
- Once the topic is comprehensively covered (at least 3-4 different aspects)
- Insert 【🤖Research】 at the beginning of each response.
- When done, hand control back to "coordinator". Focus each response on one aspect.
"""
    ),
)

summary_agent = Agent(
    name="summary_agent",
    client=chat_client,
    description="summary_agent",
    tools=[mcp_tool],
    instructions=(
"""            
Summarize the research results. Provide a concise and well-organized summary.
- Insert 【🤖Summary】 at the beginning of each response.
- Perform the summary only once at the end.
- When done, hand control back to "coordinator".
"""
    ),
)

In [ ]:
# 4. -----------------  Assemble Team -----------------  
# Build workflow using HandoffBuilder
workflow = (
    HandoffBuilder(
        name="autonomous_iteration_handoff",
        participants=[coordinator, research_agent, summary_agent],
    )
    .with_start_agent(coordinator)
    .add_handoff(coordinator, [research_agent, summary_agent])
    .add_handoff(research_agent, [coordinator])  # research_agent can return to coordinator
    .add_handoff(summary_agent, [coordinator])
    # with_autonomous_mode: Mode where agents process tasks autonomously
    # turn_limits: Autonomous turn limit per agent (safety mechanism to prevent infinite loops)
    .with_autonomous_mode(turn_limits={"coordinator": 5, "research_agent": 5, "summary_agent": 5})
    # with_termination_condition: Semantic judgment of task completion (ideal termination condition)
    # Coordinator responds 3 times = research and summary are deemed complete
    .with_termination_condition(
        lambda conv: sum(1 for msg in conv if msg.author_name == "coordinator" and msg.role == "assistant")
        >= 3
    )
    .build()
)

In [ ]:
# Visualize the workflow
svg_path = visualize_workflow(
    workflow, 
    filename="handoff_workflow",
    show_preview=True
)


- `with_autonomous_mode()`: Sets the workflow to autonomous mode. In autonomous mode, agents (including specialists) continue iterating on tasks until they explicitly call the handoff tool or reach the turn limit. This allows specialists to execute long-running autonomous tasks (research, coding, analysis, etc.) without needing to return control to the coordinator or user early. If `with_autonomous_mode()` is not called, the default is human-in-the-loop mode (waiting for user input after agent response).

  - `turn_limits`: Specify the maximum number of autonomous turns per agent as a dictionary (e.g., `{"agent_name": 5}`). This serves as a safety mechanism to prevent infinite loops.


## Execute Handoff

In [ ]:
# task = "Please check the shipping status for user ID: 123."
task = "Please briefly describe 3 new features of Azure AI Search in 2025."
final_conversation = await run_stream_pretty(workflow, task, tracer=tracer, span_name="Handoff")

In [ ]:
for msg in final_conversation:
    print(f"⭐️Role: {msg.role}, 👤Author: {msg.author_name}, 📝Text: {msg.text[:100]}...")

In [ ]:
for msg in final_conversation:
    print(msg.author_name)

This cell **assembles the autonomous handoff workflow with 3 agents**.

Essentially, what it does:

1. **Define agent transition routes** — Build a graph structure of who can hand off work to whom
   - `coordinator` → `research_agent` / `summary_agent` (delegate research & summary)
   - `research_agent` → `coordinator` (return results)
   - `summary_agent` → `coordinator` (return results)

2. **Autonomous mode setting** — Automatically transition between agents without user input. Each agent is capped at 5 turns

3. **Termination condition** — Considered complete when `coordinator` responds 3 times as assistant (= the cycle of research instruction → research result received → summary result received has completed)

In short, this is an orchestration blueprint where **"the coordinator acts as the command center, delegates work in the order of research → summary, and automatically terminates when all results are gathered"**.

```
User Input
  ↓
coordinator (planning & routing)  ←──┐
  ↓ handoff                     │
research_agent (research) ──────────┤
  ↓ (re-delegate via coordinator)    │
summary_agent (summary) ───────────┘
  ↓
coordinator (final response) → End
```

In [ ]:
# ========================================
# 1. Agent Definitions - Simple Version
# ========================================

# Triage agent: Routes inquiries to the appropriate specialist
triage = Agent(
    name="triage",
    client=chat_client,
    instructions="""
You are a customer support triage agent.
Route customer inquiries to the appropriate specialist based on the content:

- Inquiries about order status or delivery → order_specialist
- Inquiries about returns or exchanges → return_specialist
- Inquiries about billing or payments → billing_specialist

Perform routing immediately using the handoff tool. No explanation is needed.
""",
)

# Order specialist
order_specialist = Agent(
    name="order_specialist",
    client=chat_client,
    tools=[mcp_tool],
    instructions="""
You are an order and delivery specialist.
Always verify data using mcp_tools before responding. Speculative answers are prohibited.

- If the user provides an order number (order_id): First call get_shipping_status_by_order_id
- If only a user ID is available: Call get_shipping_status
- Only ask for additional information when the retrieved result is empty
- If the tool result contains data, do not say "not found" — summarize and return the results
- End when complete.
""",
)

# Return specialist
return_specialist = Agent(
    name="return_specialist",
    client=chat_client,
    tools=[mcp_tool],
    instructions="Process return and exchange requests. End when complete.",
)

# Billing specialist
billing_specialist = Agent(
    name="billing_specialist",
    client=chat_client,
    tools=[mcp_tool],
    instructions="Resolve billing and payment issues. End when complete.",
)

---

# Handoff with User Input - Interactive Scenario

**Purpose**: Enabling user-agent interaction

**Scenario**:
1. User makes the initial inquiry
2. Agent asks questions or requests confirmation
3. **User provides additional information**
4. Agent continues processing

**モード**: `human-in-the-loop` - ユーザーの入力を待つ

In [ ]:
# ========================================
# 1. Build Interactive Workflow
# ========================================

interactive_workflow = (
    HandoffBuilder(
        name="interactive_handoff",
        participants=[triage, order_specialist, return_specialist, billing_specialist],
    )
    .with_start_agent(triage)
    .add_handoff(triage, [order_specialist, return_specialist, billing_specialist])
    .add_handoff(order_specialist, [triage])
    .add_handoff(return_specialist, [triage])
    .add_handoff(billing_specialist, [triage])
    
    # human_in_loop mode: Not calling with_autonomous_mode() defaults to interactive mode
    # After agent response, waits for user input
    
    # Termination condition: End when user types "done", "end", or "thank you"
    .with_termination_condition(
        lambda conv: any(
            msg.role == "user" and msg.text and ("done" in msg.text.lower() or "end" in msg.text.lower() or "thank" in msg.text.lower())
            for msg in conv
        )
    )
    .build()
)

print("✅ 対話型Handoffワークフローを構築しました")

In [ ]:
# Visualize the workflow
svg_path = visualize_workflow(
    interactive_workflow, 
    filename="interactive_handoff_workflow",
    show_preview=True
)

In [ ]:

# ========================================
# 2. Execution Function That Accepts User Input
# ========================================

import sys
from agent_framework import AgentResponseUpdate, Message, WorkflowEvent

def _as_text(data: object) -> str:
    if data is None:
        return ""
    if isinstance(data, str):
        return data
    text = getattr(data, "text", None)
    if isinstance(text, str):
        return text
    return str(data)

def _print_tool_usage_summary(conversation: list[Message]) -> None:
    tool_msgs = [
        msg for msg in conversation
        if getattr(msg, "role", None) == "tool"
    ]
    print(f"🔎 Number of tool messages: {len(tool_msgs)}")
    if tool_msgs:
        latest = getattr(tool_msgs[-1], "text", "")
        if latest:
            preview = latest if len(latest) <= 240 else latest[:240] + "..."
            print(f"🔎 Latest tool result (beginning): {preview}")

async def run_interactive_handoff(workflow, initial_message: str):
    """
    Interactive Handoff execution - Accepts user input

    ★ Important: Do not break/return within the async for loop.
      If the async generator returned by workflow.run() is abandoned midway,
      a GeneratorExit occurs in the framework's internal _run_workflow_with_tracing(),
      causing OpenTelemetry context detach to fail.
      Always consume the generator to completion and control via flags.
    """
    print(f"\n{'='*60}")
    print(f"💬 ユーザー: {initial_message}")
    print(f"{'='*60}\n")

    # Scripted input (for Notebook)
    scripted_input = [
        "The order number is 840905",
        "Yes, please",
        "Thank you very much"
    ]
    input_index = 0

    current_agent = None
    pending_requests = []
    final_conversation: list[Message] = []

    # ★ Nest workflow internal spans as children using start_as_current_span()
    cm = tracer.start_as_current_span("Handoff") if tracer else nullcontext()
    with cm:
        # Initial execution — consume to the end without break
        async for event in workflow.run(initial_message, stream=True):
            if not isinstance(event, WorkflowEvent):
                continue
            if event.type == "output":
                if isinstance(event.data, AgentResponseUpdate):
                    agent_name = event.executor_id
                    if agent_name != current_agent:
                        if current_agent is not None:
                            sys.stdout.write("\n\n")
                            sys.stdout.flush()
                        sys.stdout.write(f"🤖 {agent_name}: ")
                        sys.stdout.flush()
                        current_agent = agent_name
                    chunk = event.data.text
                    if chunk:
                        sys.stdout.write(chunk)
                        sys.stdout.flush()
                elif isinstance(event.data, list):
                    final_conversation = cast(list[Message], event.data)
                    # ★ Do not break — let the generator be consumed to the end
            elif event.type == "request_info":
                pending_requests.append(event)

        # Complete if final_conversation is already set
        if final_conversation:
            print("\n" + "="*60)
            print("✅ Workflow complete")
            _print_tool_usage_summary(final_conversation)
            print("="*60)
            return final_conversation

        # Loop to accept user input
        while pending_requests:
            print("\n" + "-"*60)

            if input_index < len(scripted_input):
                user_input = scripted_input[input_index]
                input_index += 1
                print(f"💬 ユーザー: {user_input}")
            else:
                print("(Scripted input has ended)")
                break

            responses = {
                req.request_id: [Message(role="user", text=user_input)]
                for req in pending_requests
            }

            pending_requests = []
            current_agent = None

            # ★ Consume to the end without break
            async for event in workflow.run(stream=True, responses=responses):
                if not isinstance(event, WorkflowEvent):
                    continue
                if event.type == "output":
                    if isinstance(event.data, AgentResponseUpdate):
                        agent_name = event.executor_id
                        if agent_name != current_agent:
                            if current_agent is not None:
                                sys.stdout.write("\n\n")
                                sys.stdout.flush()
                            sys.stdout.write(f"🤖 {agent_name}: ")
                            sys.stdout.flush()
                            current_agent = agent_name
                        chunk = event.data.text
                        if chunk:
                            sys.stdout.write(chunk)
                            sys.stdout.flush()
                    elif isinstance(event.data, list):
                        final_conversation = cast(list[Message], event.data)
                elif event.type == "request_info":
                    pending_requests.append(event)

            # Determine if this round is complete
            if final_conversation:
                print("\n" + "="*60)
                print("✅ Workflow complete")
                _print_tool_usage_summary(final_conversation)
                print("="*60)
                return final_conversation

    print("\n" + "="*60)
    print("✅ Session ended")
    _print_tool_usage_summary(final_conversation)
    print("="*60)
    return final_conversation


## 対話型実行例

In [ ]:
# Scenario: Interactive session where agent requests additional information
ret = await run_interactive_handoff(
    interactive_workflow, 
    "I would like to check the delivery status of my order"
)

---

## 🎯 Differences Between the Two Modes

### **Autonomous Mode**
```python
.with_autonomous_mode(turn_limits={"agent_name": 10})
```
- Agents **complete tasks automatically**
- User input is **only required once at the beginning**
- Agents collaborate autonomously

**Use cases**: 
- Information retrieval tasks
- Data analysis
- Code generation

---

### **Human-in-the-loop Mode** (Interactive / Default)
```python
# Not calling with_autonomous_mode() = Default interactive mode
HandoffBuilder(...)
    .with_start_agent(triage)
    .add_handoff(...)
    .build()
```
- Agents **ask questions to the user**
- User **provides additional information**
- Resolve through repeated interaction

**Use cases**:
- Customer support
- Detailed troubleshooting
- Tasks requiring information gathering

---

### Implementation Key Points

#### **Autonomous Mode - Implementation Example in This Notebook**
```python
# Completes in a single execution
final_conversation = await run_stream_pretty(workflow, task, tracer=tracer, span_name="Handoff")
```

#### **Human-in-the-loop Mode - Implementation Example in This Notebook**
```python
ret = await run_interactive_handoff(
    interactive_workflow,
    "I would like to check the delivery status of my order"
)
```

#### **Human-in-the-loop Mode - Key Points of the Input Loop**
```python
while pending_requests:
    user_input = input("💬 あなた: ")  # または scripted_input[i]
    responses = {
        req.request_id: [Message(role="user", text=user_input)]
        for req in pending_requests
    }
    async for event in workflow.send_responses_streaming(responses):
        ...
```

**使い分け**: タスクの性質に応じて適切なモードを選択 🎯